In [3]:
import requests
import json
import pandas as pd
from pathlib import Path

# SEC request headers
headers = {
    "User-Agent": "HealthcareResearch contact: kishanpatel6089@gmail.com"
}

# Pfizer CIK
pfizer_cik = "0000078003"

# SEC Company Facts API
pfizer_url = (
    f"https://data.sec.gov/api/xbrl/companyfacts/CIK{pfizer_cik}.json"
)

pfizer_response = requests.get(
    pfizer_url,
    headers=headers
)

print("Status:", pfizer_response.status_code)

pfizer_data = pfizer_response.json()

print("Company:", pfizer_data["entityName"])


Status: 200
Company: PFIZER INC


In [5]:

# Extract all US-GAAP tags from Pfizer SEC data

facts = pfizer_data["facts"]["us-gaap"]

print("Total unique US-GAAP tags:", len(facts))

# Create tag catalogue
tag_records = []

for tag_name, details in facts.items():

    units = details.get("units", {})

    if not units:
        tag_records.append({
            "Tag": tag_name,
            "Label": details.get("label", ""),
            "Description": details.get("description", ""),
            "Unit": "",
            "Available_Years": ""
        })

    for unit_name, observations in units.items():

        years = sorted({
            item.get("fy")
            for item in observations
            if item.get("fy") is not None
        })

        tag_records.append({
            "Tag": tag_name,
            "Label": details.get("label", ""),
            "Description": details.get("description", ""),
            "Unit": unit_name,
            "Available_Years": ", ".join(map(str, years))
        })

pfizer_tag_catalog = pd.DataFrame(tag_records)

# Save to raw data folder
output_path = Path("../data/raw/pfizer_usgaap_tag_catalog.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

pfizer_tag_catalog.to_csv(output_path, index=False)

print("Catalogue rows:", len(pfizer_tag_catalog))
print("Unique tags in catalogue:", pfizer_tag_catalog["Tag"].nunique())
print("Saved to:", output_path)


Total unique US-GAAP tags: 668
Catalogue rows: 672
Unique tags in catalogue: 668
Saved to: ..\data\raw\pfizer_usgaap_tag_catalog.csv


In [6]:

# Identify potentially relevant financial statement tags

financial_keywords = [
    "Revenue",
    "Sales",
    "CostOfGoods",
    "ResearchAndDevelopment",
    "SellingGeneralAndAdministrative",
    "GrossProfit",
    "OperatingIncome",
    "NetIncome",
    "CashProvidedByUsedInOperatingActivities",
    "PaymentsToAcquirePropertyPlantAndEquipment"
]

financial_tags = pfizer_tag_catalog[
    pfizer_tag_catalog["Tag"].str.contains(
        "|".join(financial_keywords),
        case=False,
        na=False
    )
].copy()

financial_tags = financial_tags.sort_values("Tag")

display(financial_tags[
    ["Tag", "Label", "Unit", "Available_Years"]
])

print("Filtered catalogue rows:", len(financial_tags))


,Tag,Label,Unit,Available_Years
10,AdjustmentsNoncashItemsToReconcileNetIncomeLos...,"Adjustments, Noncash Items, to Reconcile Net I...",USD,"2009, 2010"
31,AvailableForSaleSecuritiesAndHeldToMaturitySec...,Available-for-sale Securities and Held-to-matu...,USD,"2014, 2017, 2018"
32,AvailableForSaleSecuritiesCurrent,"Available-for-sale Securities, Current",USD,2017
33,AvailableForSaleSecuritiesDebtSecuritiesCurrent,"Debt Securities, Available-for-sale, Current",USD,"2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025..."
34,AvailableForSaleSecuritiesDebtSecuritiesNoncur...,"Debt Securities, Available-for-sale, Noncurrent",USD,"2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025..."
35,AvailableForSaleSecuritiesEquitySecurities,"Available-for-sale Securities, Equity Securities",USD,"2017, 2018"
36,AvailableForSaleSecuritiesGrossRealizedGains,"Available-for-sale Securities, Gross Realized ...",USD,"2010, 2011, 2012, 2013"
37,AvailableForSaleSecuritiesGrossRealizedGainsLo...,"Available-for-sale Securities, Gross Realized ...",USD,"2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017"
38,AvailableForSaleSecuritiesGrossRealizedLosses,"Available-for-sale Securities, Gross Realized ...",USD,"2010, 2011, 2012, 2013"
39,AvailableForSaleSecuritiesNoncurrent,"Available-for-sale Securities, Noncurrent",USD,2017


Filtered catalogue rows: 55


In [7]:

# Inspect actual SEC observations for key financial tags

selected_tags = [
    "Revenues",
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "CostOfGoodsAndServicesSold",
    "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost",
    "SellingGeneralAndAdministrativeExpense",
    "NetIncomeLoss",
    "NetCashProvidedByUsedInOperatingActivities",
    "PaymentsToAcquirePropertyPlantAndEquipment"
]

for tag in selected_tags:

    if tag not in facts:
        print(f"\n{tag}: NOT FOUND")
        continue

    print(f"\n{'='*80}")
    print("TAG:", tag)
    print("LABEL:", facts[tag].get("label"))

    for unit, observations in facts[tag]["units"].items():

        df = pd.DataFrame(observations)

        if "form" in df.columns:
            df = df[df["form"] == "10-K"]

        columns = [
            col for col in
            ["start", "end", "fy", "fp", "form", "filed", "frame", "val"]
            if col in df.columns
        ]

        print("UNIT:", unit)
        display(
            df[columns]
            .sort_values("end")
            .tail(15)
        )



TAG: Revenues
LABEL: Revenues
UNIT: USD


,start,end,fy,fp,form,filed,frame,val
0,2020-01-01,2020-12-31,2022,FY,10-K,2023-02-23,CY2020,41651000000
1,2021-01-01,2021-12-31,2022,FY,10-K,2023-02-23,NaN,81288000000
2,2021-01-01,2021-12-31,2023,FY,10-K,2024-02-22,CY2021,81288000000
8,2022-01-01,2022-12-31,2022,FY,10-K,2023-02-23,NaN,100330000000
9,2022-01-01,2022-12-31,2023,FY,10-K,2024-02-22,NaN,100330000000
10,2022-01-01,2022-12-31,2024,FY,10-K,2025-02-27,CY2022,101175000000
21,2023-01-01,2023-12-31,2023,FY,10-K,2024-02-22,NaN,58496000000
22,2023-01-01,2023-12-31,2024,FY,10-K,2025-02-27,NaN,59553000000
23,2023-01-01,2023-12-31,2025,FY,10-K,2026-02-26,CY2023,59553000000
34,2024-01-01,2024-12-31,2024,FY,10-K,2025-02-27,NaN,63627000000



TAG: RevenueFromContractWithCustomerExcludingAssessedTax
LABEL: Revenue from Contract with Customer, Excluding Assessed Tax
UNIT: USD


,start,end,fy,fp,form,filed,frame,val
0,2016-01-01,2016-12-31,2018,FY,10-K,2019-02-28,CY2016,52824000000
5,2017-01-01,2017-12-31,2018,FY,10-K,2019-02-28,NaN,52546000000
6,2017-01-01,2017-12-31,2019,FY,10-K,2020-02-27,CY2017,52546000000
16,2018-01-01,2018-12-31,2018,FY,10-K,2019-02-28,NaN,53647000000
17,2018-01-01,2018-12-31,2019,FY,10-K,2020-02-27,NaN,53647000000
18,2018-01-01,2018-12-31,2020,FY,10-K,2021-02-25,CY2018,40825000000
29,2019-01-01,2019-12-31,2019,FY,10-K,2020-02-27,NaN,51750000000
30,2019-01-01,2019-12-31,2020,FY,10-K,2021-02-25,NaN,41172000000
31,2019-01-01,2019-12-31,2021,FY,10-K,2022-02-24,CY2019,40905000000
42,2020-01-01,2020-12-31,2020,FY,10-K,2021-02-25,NaN,41908000000



TAG: CostOfGoodsAndServicesSold
LABEL: Cost of Goods and Services Sold
UNIT: USD


,start,end,fy,fp,form,filed,frame,val
107,2020-01-01,2020-12-31,2020,FY,10-K,2021-02-25,NaN,8692000000
108,2020-01-01,2020-12-31,2021,FY,10-K,2022-02-24,NaN,8484000000
109,2020-01-01,2020-12-31,2022,FY,10-K,2023-02-23,CY2020,8484000000
121,2021-01-01,2021-12-31,2022,FY,10-K,2023-02-23,NaN,30821000000
122,2021-01-01,2021-12-31,2023,FY,10-K,2024-02-22,CY2021,30821000000
120,2021-01-01,2021-12-31,2021,FY,10-K,2022-02-24,NaN,30821000000
133,2022-01-01,2022-12-31,2022,FY,10-K,2023-02-23,NaN,34344000000
134,2022-01-01,2022-12-31,2023,FY,10-K,2024-02-22,NaN,34344000000
135,2022-01-01,2022-12-31,2024,FY,10-K,2025-02-27,CY2022,34344000000
146,2023-01-01,2023-12-31,2023,FY,10-K,2024-02-22,NaN,24954000000


... (6 outputs left out)


TAG: NetCashProvidedByUsedInOperatingActivities
LABEL: Net Cash Provided by (Used in) Operating Activities
UNIT: USD


,start,end,fy,fp,form,filed,frame,val
110,2020-01-01,2020-12-31,2021,FY,10-K,2022-02-24,NaN,14403000000
111,2020-01-01,2020-12-31,2022,FY,10-K,2023-02-23,CY2020,14403000000
109,2020-01-01,2020-12-31,2020,FY,10-K,2021-02-25,NaN,14403000000
118,2021-01-01,2021-12-31,2021,FY,10-K,2022-02-24,NaN,32580000000
119,2021-01-01,2021-12-31,2022,FY,10-K,2023-02-23,NaN,32580000000
120,2021-01-01,2021-12-31,2023,FY,10-K,2024-02-22,CY2021,32580000000
127,2022-01-01,2022-12-31,2022,FY,10-K,2023-02-23,NaN,29267000000
128,2022-01-01,2022-12-31,2023,FY,10-K,2024-02-22,NaN,29267000000
129,2022-01-01,2022-12-31,2024,FY,10-K,2025-02-27,CY2022,29267000000
136,2023-01-01,2023-12-31,2023,FY,10-K,2024-02-22,NaN,8700000000



TAG: PaymentsToAcquirePropertyPlantAndEquipment
LABEL: Payments to Acquire Property, Plant, and Equipment
UNIT: USD


,start,end,fy,fp,form,filed,frame,val
110,2020-01-01,2020-12-31,2021,FY,10-K,2022-02-24,NaN,2226000000
111,2020-01-01,2020-12-31,2022,FY,10-K,2023-02-23,CY2020,2226000000
109,2020-01-01,2020-12-31,2020,FY,10-K,2021-02-25,NaN,2252000000
118,2021-01-01,2021-12-31,2021,FY,10-K,2022-02-24,NaN,2711000000
119,2021-01-01,2021-12-31,2022,FY,10-K,2023-02-23,NaN,2711000000
120,2021-01-01,2021-12-31,2023,FY,10-K,2024-02-22,CY2021,2711000000
127,2022-01-01,2022-12-31,2022,FY,10-K,2023-02-23,NaN,3236000000
128,2022-01-01,2022-12-31,2023,FY,10-K,2024-02-22,NaN,3236000000
129,2022-01-01,2022-12-31,2024,FY,10-K,2025-02-27,CY2022,3236000000
136,2023-01-01,2023-12-31,2023,FY,10-K,2024-02-22,NaN,3907000000


In [8]:

# Inspect full metadata for annual observations

audit_tags = [
    "Revenues",
    "CostOfGoodsAndServicesSold",
    "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost",
    "SellingGeneralAndAdministrativeExpense",
    "NetIncomeLoss"
]

for tag in audit_tags:

    print(f"\n{'='*90}")
    print("TAG:", tag)

    observations = facts[tag]["units"]["USD"]

    df = pd.DataFrame(observations)

    df = df[
        (df["form"] == "10-K") &
        (df["end"].between("2021-01-01", "2025-12-31"))
    ]

    audit_columns = [
        col for col in [
            "start", "end", "fy", "fp", "filed",
            "accn", "frame", "val", "segment"
        ]
        if col in df.columns
    ]

    display(
        df[audit_columns]
        .sort_values(["end", "filed"])
        .reset_index(drop=True)
    )



TAG: Revenues


,start,end,fy,fp,filed,accn,frame,val
0,2021-01-01,2021-12-31,2022,FY,2023-02-23,0000078003-23-000024,NaN,81288000000
1,2021-01-01,2021-12-31,2023,FY,2024-02-22,0000078003-24-000039,CY2021,81288000000
2,2022-01-01,2022-12-31,2022,FY,2023-02-23,0000078003-23-000024,NaN,100330000000
3,2022-01-01,2022-12-31,2023,FY,2024-02-22,0000078003-24-000039,NaN,100330000000
4,2022-01-01,2022-12-31,2024,FY,2025-02-27,0000078003-25-000054,CY2022,101175000000
5,2023-01-01,2023-12-31,2023,FY,2024-02-22,0000078003-24-000039,NaN,58496000000
6,2023-01-01,2023-12-31,2024,FY,2025-02-27,0000078003-25-000054,NaN,59553000000
7,2023-01-01,2023-12-31,2025,FY,2026-02-26,0000078003-26-000026,CY2023,59553000000
8,2024-01-01,2024-12-31,2024,FY,2025-02-27,0000078003-25-000054,NaN,63627000000
9,2024-01-01,2024-12-31,2025,FY,2026-02-26,0000078003-26-000026,CY2024,63627000000



TAG: CostOfGoodsAndServicesSold


,start,end,fy,fp,filed,accn,frame,val
0,2021-01-01,2021-12-31,2021,FY,2022-02-24,0000078003-22-000027,NaN,30821000000
1,2021-01-01,2021-12-31,2022,FY,2023-02-23,0000078003-23-000024,NaN,30821000000
2,2021-01-01,2021-12-31,2023,FY,2024-02-22,0000078003-24-000039,CY2021,30821000000
3,2022-01-01,2022-12-31,2022,FY,2023-02-23,0000078003-23-000024,NaN,34344000000
4,2022-01-01,2022-12-31,2023,FY,2024-02-22,0000078003-24-000039,NaN,34344000000
5,2022-01-01,2022-12-31,2024,FY,2025-02-27,0000078003-25-000054,CY2022,34344000000
6,2023-01-01,2023-12-31,2023,FY,2024-02-22,0000078003-24-000039,NaN,24954000000
7,2023-01-01,2023-12-31,2024,FY,2025-02-27,0000078003-25-000054,NaN,24954000000
8,2023-01-01,2023-12-31,2025,FY,2026-02-26,0000078003-26-000026,CY2023,24954000000
9,2024-01-01,2024-12-31,2024,FY,2025-02-27,0000078003-25-000054,NaN,17851000000



TAG: ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost


,start,end,fy,fp,filed,accn,frame,val
0,2021-01-01,2021-12-31,2021,FY,2022-02-24,0000078003-22-000027,NaN,13829000000
1,2021-01-01,2021-12-31,2022,FY,2023-02-23,0000078003-23-000024,NaN,10360000000
2,2021-01-01,2021-12-31,2023,FY,2024-02-22,0000078003-24-000039,CY2021,10360000000
3,2022-01-01,2022-12-31,2022,FY,2023-02-23,0000078003-23-000024,NaN,11428000000
4,2022-01-01,2022-12-31,2023,FY,2024-02-22,0000078003-24-000039,NaN,11428000000
5,2022-01-01,2022-12-31,2024,FY,2025-02-27,0000078003-25-000054,CY2022,11428000000
6,2023-01-01,2023-12-31,2023,FY,2024-02-22,0000078003-24-000039,NaN,10679000000
7,2023-01-01,2023-12-31,2024,FY,2025-02-27,0000078003-25-000054,NaN,10679000000
8,2023-01-01,2023-12-31,2025,FY,2026-02-26,0000078003-26-000026,CY2023,10679000000
9,2024-01-01,2024-12-31,2024,FY,2025-02-27,0000078003-25-000054,NaN,10822000000



TAG: SellingGeneralAndAdministrativeExpense


,start,end,fy,fp,filed,accn,frame,val
0,2021-01-01,2021-12-31,2021,FY,2022-02-24,0000078003-22-000027,NaN,12703000000
1,2021-01-01,2021-12-31,2022,FY,2023-02-23,0000078003-23-000024,NaN,12703000000
2,2021-01-01,2021-12-31,2023,FY,2024-02-22,0000078003-24-000039,CY2021,12703000000
3,2022-01-01,2022-12-31,2022,FY,2023-02-23,0000078003-23-000024,NaN,13677000000
4,2022-01-01,2022-12-31,2023,FY,2024-02-22,0000078003-24-000039,NaN,13677000000
5,2022-01-01,2022-12-31,2024,FY,2025-02-27,0000078003-25-000054,CY2022,13677000000
6,2023-01-01,2023-12-31,2023,FY,2024-02-22,0000078003-24-000039,NaN,14771000000
7,2023-01-01,2023-12-31,2024,FY,2025-02-27,0000078003-25-000054,NaN,14771000000
8,2023-01-01,2023-12-31,2025,FY,2026-02-26,0000078003-26-000026,CY2023,14771000000
9,2024-01-01,2024-12-31,2024,FY,2025-02-27,0000078003-25-000054,NaN,14730000000



TAG: NetIncomeLoss


,start,end,fy,fp,filed,accn,frame,val
0,2021-01-01,2021-12-31,2021,FY,2022-02-24,0000078003-22-000027,NaN,21979000000
1,2021-01-01,2021-12-31,2022,FY,2023-02-23,0000078003-23-000024,NaN,21979000000
2,2021-01-01,2021-12-31,2023,FY,2024-02-22,0000078003-24-000039,CY2021,21979000000
3,2022-01-01,2022-12-31,2022,FY,2023-02-23,0000078003-23-000024,NaN,31372000000
4,2022-01-01,2022-12-31,2023,FY,2024-02-22,0000078003-24-000039,NaN,31372000000
5,2022-01-01,2022-12-31,2024,FY,2025-02-27,0000078003-25-000054,CY2022,31372000000
6,2023-01-01,2023-12-31,2023,FY,2024-02-22,0000078003-24-000039,NaN,2119000000
7,2023-01-01,2023-12-31,2024,FY,2025-02-27,0000078003-25-000054,NaN,2119000000
8,2023-01-01,2023-12-31,2025,FY,2026-02-26,0000078003-26-000026,CY2023,2119000000
9,2024-01-01,2024-12-31,2024,FY,2025-02-27,0000078003-25-000054,NaN,8031000000


In [9]:

# Select financial tags for FY2021-FY2025

selected_tags = {
    "Revenue": "Revenues",
    "COGS": "CostOfGoodsAndServicesSold",
    "R&D": "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost",
    "SG&A": "SellingGeneralAndAdministrativeExpense",
    "Net_Income": "NetIncomeLoss",
    "Operating_Cash_Flow": "NetCashProvidedByUsedInOperatingActivities",
    "Capital_Expenditure": "PaymentsToAcquirePropertyPlantAndEquipment"
}

financial_records = []

for metric, tag in selected_tags.items():

    observations = facts[tag]["units"]["USD"]
    df = pd.DataFrame(observations)

    # Keep annual 10-K observations for calendar years 2021-2025
    df = df[
        (df["form"] == "10-K") &
        (df["fp"] == "FY") &
        (df["start"] == df["end"].str[:4] + "-01-01") &
        (df["end"].str[:4].astype(int).between(2021, 2025)) &
        (df["end"].str.endswith("12-31"))
    ].copy()

    # Select latest filed observation for each reporting year
    df = (
        df.sort_values("filed")
        .drop_duplicates(subset=["end"], keep="last")
    )

    for _, row in df.iterrows():
        financial_records.append({
            "Year": int(row["end"][:4]),
            "Metric": metric,
            "Value_USD": row["val"],
            "Value_USD_Bn": row["val"] / 1_000_000_000,
            "Filed_Date": row["filed"],
            "Accession_Number": row["accn"]
        })

pfizer_financial_long = pd.DataFrame(financial_records)

# Convert to a year-by-year financial table
pfizer_financial = pfizer_financial_long.pivot(
    index="Year",
    columns="Metric",
    values="Value_USD_Bn"
).reset_index()

display(pfizer_financial.round(3))


Metric,Year,COGS,Capital_Expenditure,Net_Income,Operating_Cash_Flow,R&D,Revenue,SG&A
0,2021,30.821,2.711,21.979,32.580,10.360,81.288,12.703
1,2022,34.344,3.236,31.372,29.267,11.428,101.175,13.677
2,2023,24.954,3.907,2.119,8.700,10.679,59.553,14.771
3,2024,17.851,2.909,8.031,12.744,10.822,63.627,14.730
4,2025,16.067,2.629,7.771,11.704,10.437,62.579,13.794


In [10]:

# Inspect all FY2021 R&D observations

tag = "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost"

rd_data = pd.DataFrame(
    facts[tag]["units"]["USD"]
)

rd_2021 = rd_data[
    (rd_data["start"] == "2021-01-01") &
    (rd_data["end"] == "2021-12-31")
]

display(
    rd_2021[
        ["fy", "fp", "form", "filed", "accn", "frame", "val"]
    ].sort_values("filed")
)


,fy,fp,form,filed,accn,frame,val
172,2021,FY,10-K,2022-02-24,0000078003-22-000027,NaN,13829000000
173,2022,FY,10-K,2023-02-23,0000078003-23-000024,NaN,10360000000
174,2023,FY,10-K,2024-02-22,0000078003-24-000039,CY2021,10360000000


In [11]:

# Validate FY2021 Pfizer R&D reclassification

rd_original_2021 = 13.829
rd_reclassified_2021 = 10.360
acquired_iprd_2021 = 3.469

difference = rd_original_2021 - (
    rd_reclassified_2021 + acquired_iprd_2021
)

print("Original FY2021 R&D:", rd_original_2021, "USD bn")
print("Reclassified R&D:", rd_reclassified_2021, "USD bn")
print("Acquired IPR&D:", acquired_iprd_2021, "USD bn")
print("Reconciliation difference:", round(difference, 6), "USD bn")

assert abs(difference) < 0.000001


Original FY2021 R&D: 13.829 USD bn
Reclassified R&D: 10.36 USD bn
Acquired IPR&D: 3.469 USD bn
Reconciliation difference: 0.0 USD bn


In [12]:

# Calculate Pfizer financial performance metrics

pfizer_financial["Gross_Profit"] = (
    pfizer_financial["Revenue"] - pfizer_financial["COGS"]
)

pfizer_financial["Gross_Margin_%"] = (
    pfizer_financial["Gross_Profit"] /
    pfizer_financial["Revenue"] * 100
)

pfizer_financial["Revenue_Growth_%"] = (
    pfizer_financial["Revenue"].pct_change() * 100
)

pfizer_financial["R&D_Intensity_%"] = (
    pfizer_financial["R&D"] /
    pfizer_financial["Revenue"] * 100
)

pfizer_financial["SG&A_Ratio_%"] = (
    pfizer_financial["SG&A"] /
    pfizer_financial["Revenue"] * 100
)

pfizer_financial["Net_Margin_%"] = (
    pfizer_financial["Net_Income"] /
    pfizer_financial["Revenue"] * 100
)

pfizer_financial["OCF_Margin_%"] = (
    pfizer_financial["Operating_Cash_Flow"] /
    pfizer_financial["Revenue"] * 100
)

pfizer_financial["Free_Cash_Flow"] = (
    pfizer_financial["Operating_Cash_Flow"] -
    pfizer_financial["Capital_Expenditure"]
)

pfizer_financial["FCF_Margin_%"] = (
    pfizer_financial["Free_Cash_Flow"] /
    pfizer_financial["Revenue"] * 100
)

# FY2021-FY2025 revenue CAGR
revenue_cagr = (
    (pfizer_financial["Revenue"].iloc[-1] /
     pfizer_financial["Revenue"].iloc[0]) ** (1/4) - 1
) * 100

print("FY2021-FY2025 Revenue CAGR:", round(revenue_cagr, 2), "%")

display(pfizer_financial.round(2))


FY2021-FY2025 Revenue CAGR: -6.33 %


Metric,Year,COGS,Capital_Expenditure,Net_Income,Operating_Cash_Flow,R&D,Revenue,SG&A,Gross_Profit,Gross_Margin_%,Revenue_Growth_%,R&D_Intensity_%,SG&A_Ratio_%,Net_Margin_%,OCF_Margin_%,Free_Cash_Flow,FCF_Margin_%
0,2021,30.82,2.71,21.98,32.58,10.36,81.29,12.70,50.47,62.08,NaN,12.74,15.63,27.04,40.08,29.87,36.74
1,2022,34.34,3.24,31.37,29.27,11.43,101.18,13.68,66.83,66.05,24.46,11.30,13.52,31.01,28.93,26.03,25.73
2,2023,24.95,3.91,2.12,8.70,10.68,59.55,14.77,34.60,58.10,-41.14,17.93,24.80,3.56,14.61,4.79,8.05
3,2024,17.85,2.91,8.03,12.74,10.82,63.63,14.73,45.78,71.94,6.84,17.01,23.15,12.62,20.03,9.84,15.46
4,2025,16.07,2.63,7.77,11.70,10.44,62.58,13.79,46.51,74.33,-1.65,16.68,22.04,12.42,18.70,9.08,14.50


In [13]:

# Find operating income and expense-related tags

operating_tags = []

for tag, details in facts.items():
    if any(keyword in tag.lower() for keyword in [
        "operatingincome",
        "incomefromoperations",
        "operatingexpenses"
    ]):
        operating_tags.append({
            "Tag": tag,
            "Label": details.get("label", "")
        })

operating_tag_df = pd.DataFrame(operating_tags)

display(operating_tag_df)


,Tag,Label
0,OtherNonoperatingIncomeExpense,Other Nonoperating Income (Expense)
1,OtherOperatingIncomeExpenseNet,"Other Operating Income (Expense), Net"


In [14]:

# Inspect income statement-related US-GAAP tags

income_keywords = [
    "Expense",
    "IncomeLoss",
    "Profit",
    "CostOf"
]

income_statement_tags = []

for tag, details in facts.items():
    if any(keyword.lower() in tag.lower()
           for keyword in income_keywords):

        income_statement_tags.append({
            "Tag": tag,
            "Label": details.get("label", "")
        })

income_statement_df = pd.DataFrame(income_statement_tags)

display(income_statement_df.sort_values("Tag").reset_index(drop=True))

print("Total matching tags:", len(income_statement_df))


,Tag,Label
0,AccumulatedOtherComprehensiveIncomeLossNetOfTax,"Accumulated Other Comprehensive Income (Loss),..."
1,AdjustmentsNoncashItemsToReconcileNetIncomeLos...,"Adjustments, Noncash Items, to Reconcile Net I..."
2,AdvertisingExpense,Advertising Expense
3,AllocatedShareBasedCompensationExpense,"Share-based Payment Arrangement, Expense"
4,AllocatedShareBasedCompensationExpenseNetOfTax,"Share-based Payment Arrangement, Expense, afte..."
...,...,...
119,SellingGeneralAndAdministrativeExpense,"Selling, General and Administrative Expense"
120,StateAndLocalIncomeTaxExpenseBenefitContinuing...,"State and Local Income Tax Expense (Benefit), ..."
121,TaxCutsAndJobsActOf2017IncomeTaxExpenseBenefit,"Tax Cuts and Jobs Act, Income Tax Expense (Ben..."
122,TaxCutsAndJobsActOf2017MeasurementPeriodAdjust...,"Tax Cuts and Jobs Act, Measurement Period Adju..."


Total matching tags: 124


In [15]:

# Filter potentially relevant operating income and expense tags

operating_candidates = income_statement_df[
    income_statement_df["Tag"].str.contains(
        "Operating|ProfitLoss|IncomeLossFromContinuing",
        case=False,
        na=False
    )
].copy()

display(
    operating_candidates.sort_values("Tag").to_string(index=False)
)


'                                                                                                        Tag                                                                                                                                        Label\n              AdjustmentsNoncashItemsToReconcileNetIncomeLossToCashProvidedByUsedInOperatingActivitiesOther Adjustments, Noncash Items, to Reconcile Net Income (Loss) to Cash Provided by (Used in) Operating Activities, Other (Deprecated 2011-01-31)\n                                                                         IncomeLossFromContinuingOperations                                                                 Income (Loss) from Continuing Operations, Net of Tax, Attributable to Parent\n                                       IncomeLossFromContinuingOperationsAttributableToNoncontrollingEntity                                                Income (Loss) from Continuing Operations, Net of Tax, Attributable to Noncontrolling Interest\

In [16]:

# Retrieve Pfizer's latest 10-K filing information

submissions_url = (
    f"https://data.sec.gov/submissions/CIK{pfizer_cik}.json"
)

submissions_response = requests.get(
    submissions_url,
    headers=headers
)

submissions_data = submissions_response.json()

recent_filings = pd.DataFrame(
    submissions_data["filings"]["recent"]
)

pfizer_10k = recent_filings[
    recent_filings["form"] == "10-K"
].head(5)

display(
    pfizer_10k[
        ["filingDate", "accessionNumber", "primaryDocument"]
    ]
)


,filingDate,accessionNumber,primaryDocument
99,2026-02-26,0000078003-26-000026,pfe-20251231.htm
228,2025-02-27,0000078003-25-000054,pfe-20241231.htm
415,2024-02-22,0000078003-24-000039,pfe-20231231.htm
607,2023-02-23,0000078003-23-000024,pfe-20221231.htm
790,2022-02-24,0000078003-22-000027,pfe-20211231.htm


In [17]:

# Download Pfizer FY2025 Form 10-K

pfizer_10k_url = (
    "https://www.sec.gov/Archives/edgar/data/78003/"
    "000007800326000026/pfe-20251231.htm"
)

pfizer_10k_response = requests.get(
    pfizer_10k_url,
    headers=headers
)

print("Status:", pfizer_10k_response.status_code)

from bs4 import BeautifulSoup

pfizer_soup = BeautifulSoup(
    pfizer_10k_response.content,
    "html.parser"
)

print("Page title:", pfizer_soup.title.get_text(strip=True))
print("Tables found:", len(pfizer_soup.find_all("table")))


Status: 200
Page title: pfe-20251231
Tables found: 315


In [18]:

# Identify income statement tables in Pfizer's 2025 10-K

all_tables = pfizer_soup.find_all("table")

for i, table in enumerate(all_tables):

    table_text = table.get_text(" ", strip=True)

    if (
        "Total revenues" in table_text
        and "Cost of sales" in table_text
        and "Research and development expenses" in table_text
    ):
        print("\nTable index:", i)
        print(table_text[:5000])



Table index: 84
Costs and expenses follow: Year Ended December 31, % Change (MILLIONS) 2025 2024 2023 25/24 24/23 Cost of sales $ 16,067 $ 17,851 $ 24,954 (10) (28) Percentage of Total revenues 25.7 % 28.1 % 41.9 % Selling, informational and administrative expenses 13,794 14,730 14,771 (6) — Research and development expenses 10,437 10,822 10,679 (4) 1 Acquired in-process research and development expenses 1,613 108 194 * (44) Amortization of intangible assets 4,874 5,286 4,733 (8) 12 Restructuring charges and certain acquisition-related costs 1,550 2,419 2,943 (36) (18) Other (income)/deductions—net 6,724 4,388 222 53 *

Table index: 114
Year Ended December 31, (MILLIONS, EXCEPT PER SHARE DATA) 2025 2024 2023 Revenues: Product revenues $ 51,663 $ 53,816 $ 50,914 Alliance revenues 9,266 8,388 7,582 Royalty revenues 1,650 1,423 1,058 Total revenues 62,579 63,627 59,553 Costs and expenses: Cost of sales (a), (b) 16,067 17,851 24,954 Selling, informational and administrative expenses (a) 1

In [20]:

# Extract Pfizer's income statement using BeautifulSoup

table = all_tables[114]

rows = []

for tr in table.find_all("tr"):
    cells = [
        cell.get_text(" ", strip=True)
        for cell in tr.find_all(["th", "td"], recursive=False)
    ]

    if cells:
        rows.append(cells)

statement_table = pd.DataFrame(rows)

display(statement_table)



,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
0,,,,,,,,,,,...,,,,,,,,,,
1,,,,"Year Ended December 31,",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"(MILLIONS, EXCEPT PER SHARE DATA)",,,,,2025,,2024,,2023,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Revenues:,,,,,,,,,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Product revenues,,,,,$,"51,663",,,$,...,"50,914",,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Alliance revenues,,,,,"9,266",,,"8,388",,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Royalty revenues,,,,,"1,650",,,"1,423",,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Total revenues,,,,,"62,579",,,"63,627",,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Costs and expenses:,,,,,,,,,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,"Cost of sales (a), (b)",,,,,"16,067",,,"17,851",,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:

# Extract selected financial figures from Pfizer's statement

statement_text = all_tables[114].get_text(" ", strip=True)

financial_items = {
    "Revenue": [62579, 63627, 59553],
    "COGS": [16067, 17851, 24954],
    "SG&A": [13794, 14730, 14771],
    "R&D": [10437, 10822, 10679],
    "Acquired_IPR&D": [1613, 108, 194],
    "Amortization": [4874, 5286, 4733],
    "Restructuring": [1550, 2419, 2943],
    "Other_Income_Deductions": [6724, 4388, 222],
    "Income_Before_Tax": [7520, 8023, 1058]
}

pfizer_statement = pd.DataFrame(
    financial_items,
    index=[2025, 2024, 2023]
)

pfizer_statement.index.name = "Year"

# Convert USD millions to USD billions
pfizer_statement = pfizer_statement / 1000

display(pfizer_statement.round(3))


,Revenue,COGS,SG&A,R&D,Acquired_IPR&D,Amortization,Restructuring,Other_Income_Deductions,Income_Before_Tax
Year,,,,,,,,,
2025,62.579,16.067,13.794,10.437,1.613,4.874,1.550,6.724,7.520
2024,63.627,17.851,14.730,10.822,0.108,5.286,2.419,4.388,8.023
2023,59.553,24.954,14.771,10.679,0.194,4.733,2.943,0.222,1.058


In [22]:

# Add additional income statement metrics to the five-year dataset

additional_metrics = pfizer_statement[
    [
        "Acquired_IPR&D",
        "Amortization",
        "Restructuring",
        "Other_Income_Deductions",
        "Income_Before_Tax"
    ]
]

pfizer_financial = (
    pfizer_financial
    .set_index("Year")
    .join(additional_metrics, how="left")
    .reset_index()
)

display(pfizer_financial.round(3))


,Year,COGS,Capital_Expenditure,Net_Income,Operating_Cash_Flow,R&D,Revenue,SG&A,Gross_Profit,Gross_Margin_%,...,SG&A_Ratio_%,Net_Margin_%,OCF_Margin_%,Free_Cash_Flow,FCF_Margin_%,Acquired_IPR&D,Amortization,Restructuring,Other_Income_Deductions,Income_Before_Tax
0,2021,30.821,2.711,21.979,32.580,10.360,81.288,12.703,50.467,62.084,...,15.627,27.038,40.080,29.869,36.745,NaN,NaN,NaN,NaN,NaN
1,2022,34.344,3.236,31.372,29.267,11.428,101.175,13.677,66.831,66.055,...,13.518,31.008,28.927,26.031,25.729,NaN,NaN,NaN,NaN,NaN
2,2023,24.954,3.907,2.119,8.700,10.679,59.553,14.771,34.599,58.098,...,24.803,3.558,14.609,4.793,8.048,0.194,4.733,2.943,0.222,1.058
3,2024,17.851,2.909,8.031,12.744,10.822,63.627,14.730,45.776,71.944,...,23.151,12.622,20.029,9.835,15.457,0.108,5.286,2.419,4.388,8.023
4,2025,16.067,2.629,7.771,11.704,10.437,62.579,13.794,46.512,74.325,...,22.043,12.418,18.703,9.075,14.502,1.613,4.874,1.550,6.724,7.520


In [23]:

# Download Pfizer FY2024 Form 10-K

pfizer_2024_url = (
    "https://www.sec.gov/Archives/edgar/data/78003/"
    "000007800325000054/pfe-20241231.htm"
)

pfizer_2024_response = requests.get(
    pfizer_2024_url,
    headers=headers
)

print("Status:", pfizer_2024_response.status_code)

pfizer_2024_soup = BeautifulSoup(
    pfizer_2024_response.content,
    "html.parser"
)

print("Page title:", pfizer_2024_soup.title.get_text(strip=True))
print("Tables found:", len(pfizer_2024_soup.find_all("table")))


Status: 200
Page title: pfe-20241231
Tables found: 322


In [24]:

# Find Pfizer FY2024 consolidated income statement

pfizer_2024_tables = pfizer_2024_soup.find_all("table")

for i, table in enumerate(pfizer_2024_tables):
    table_text = table.get_text(" ", strip=True)

    if (
        "Total revenues" in table_text
        and "Cost of sales" in table_text
        and "Research and development expenses" in table_text
        and "Income from continuing operations before" in table_text
    ):
        print("\nTable index:", i)
        print(table_text[:3500])



Table index: 114
Year Ended December 31, (MILLIONS, EXCEPT PER SHARE DATA) 2024 2023 2022 Revenues: Product revenues $ 53,816 $ 50,914 $ 91,793 Alliance revenues 8,388 7,582 8,537 Royalty revenues (a) 1,423 1,058 845 Total revenues 63,627 59,553 101,175 Costs and expenses: Cost of sales (b), (c) 17,851 24,954 34,344 Selling, informational and administrative expenses (b) 14,730 14,771 13,677 Research and development expenses (b) 10,822 10,679 11,428 Acquired in-process research and development expenses 108 194 953 Amortization of intangible assets 5,286 4,733 3,609 Restructuring charges and certain acquisition-related costs 2,419 2,943 1,375 Other (income)/deductions––net 4,388 222 1,062 Income from continuing operations before provision/(benefit) for taxes on income 8,023 1,058 34,729 Provision/(benefit) for taxes on income ( 28 ) ( 1,115 ) 3,328 Income from continuing operations 8,051 2,172 31,401 Discontinued operations––net of tax 11 ( 15 ) 6 Net income before allocation to noncont

In [25]:

# Extract FY2022 values from Pfizer FY2024 10-K

pfizer_2022_metrics = {
    "Acquired_IPR&D": 953,
    "Amortization": 3609,
    "Restructuring": 1375,
    "Other_Income_Deductions": 1062,
    "Income_Before_Tax": 34729
}

# Convert USD millions to USD billions
pfizer_2022_metrics = {
    key: value / 1000
    for key, value in pfizer_2022_metrics.items()
}

# Add FY2022 to the existing statement
pfizer_statement.loc[2022, list(pfizer_2022_metrics)] = (
    list(pfizer_2022_metrics.values())
)

pfizer_statement = pfizer_statement.sort_index()

display(pfizer_statement.round(3))



,Revenue,COGS,SG&A,R&D,Acquired_IPR&D,Amortization,Restructuring,Other_Income_Deductions,Income_Before_Tax
Year,,,,,,,,,
2022,NaN,NaN,NaN,NaN,0.953,3.609,1.375,1.062,34.729
2023,59.553,24.954,14.771,10.679,0.194,4.733,2.943,0.222,1.058
2024,63.627,17.851,14.730,10.822,0.108,5.286,2.419,4.388,8.023
2025,62.579,16.067,13.794,10.437,1.613,4.874,1.550,6.724,7.520


In [26]:

# Add FY2021 additional income statement metrics

pfizer_2021_metrics = {
    "Acquired_IPR&D": 3.469,
    "Amortization": 3.700,
    "Restructuring": 0.802,
    "Other_Income_Deductions": -4.878,
    "Income_Before_Tax": 24.311
}

pfizer_statement.loc[2021, list(pfizer_2021_metrics)] = (
    list(pfizer_2021_metrics.values())
)

pfizer_statement = pfizer_statement.sort_index()

display(pfizer_statement.round(3))


,Revenue,COGS,SG&A,R&D,Acquired_IPR&D,Amortization,Restructuring,Other_Income_Deductions,Income_Before_Tax
Year,,,,,,,,,
2021,NaN,NaN,NaN,NaN,3.469,3.700,0.802,-4.878,24.311
2022,NaN,NaN,NaN,NaN,0.953,3.609,1.375,1.062,34.729
2023,59.553,24.954,14.771,10.679,0.194,4.733,2.943,0.222,1.058
2024,63.627,17.851,14.730,10.822,0.108,5.286,2.419,4.388,8.023
2025,62.579,16.067,13.794,10.437,1.613,4.874,1.550,6.724,7.520


In [28]:

print("Columns in pfizer_financial:")
print(pfizer_financial.columns.tolist())

print("\nColumns in pfizer_statement:")
print(pfizer_statement.columns.tolist())


Columns in pfizer_financial:
['Year', 'COGS', 'Capital_Expenditure', 'Net_Income', 'Operating_Cash_Flow', 'R&D', 'Revenue', 'SG&A', 'Gross_Profit', 'Gross_Margin_%', 'Revenue_Growth_%', 'R&D_Intensity_%', 'SG&A_Ratio_%', 'Net_Margin_%', 'OCF_Margin_%', 'Free_Cash_Flow', 'FCF_Margin_%', 'Acquired_IPR&D', 'Amortization', 'Restructuring', 'Other_Income_Deductions', 'Income_Before_Tax']

Columns in pfizer_statement:
['Revenue', 'COGS', 'SG&A', 'R&D', 'Acquired_IPR&D', 'Amortization', 'Restructuring', 'Other_Income_Deductions', 'Income_Before_Tax']


In [29]:

# Fill missing values in the existing Pfizer financial dataset

additional_columns = [
    "Acquired_IPR&D",
    "Amortization",
    "Restructuring",
    "Other_Income_Deductions",
    "Income_Before_Tax"
]

pfizer_master = pfizer_financial.set_index("Year").copy()

# Fill missing values using the statement dataset
pfizer_master.update(
    pfizer_statement[additional_columns]
)

pfizer_master = pfizer_master.reset_index()

display(pfizer_master.round(3))


,Year,COGS,Capital_Expenditure,Net_Income,Operating_Cash_Flow,R&D,Revenue,SG&A,Gross_Profit,Gross_Margin_%,...,SG&A_Ratio_%,Net_Margin_%,OCF_Margin_%,Free_Cash_Flow,FCF_Margin_%,Acquired_IPR&D,Amortization,Restructuring,Other_Income_Deductions,Income_Before_Tax
0,2021,30.821,2.711,21.979,32.580,10.360,81.288,12.703,50.467,62.084,...,15.627,27.038,40.080,29.869,36.745,3.469,3.700,0.802,-4.878,24.311
1,2022,34.344,3.236,31.372,29.267,11.428,101.175,13.677,66.831,66.055,...,13.518,31.008,28.927,26.031,25.729,0.953,3.609,1.375,1.062,34.729
2,2023,24.954,3.907,2.119,8.700,10.679,59.553,14.771,34.599,58.098,...,24.803,3.558,14.609,4.793,8.048,0.194,4.733,2.943,0.222,1.058
3,2024,17.851,2.909,8.031,12.744,10.822,63.627,14.730,45.776,71.944,...,23.151,12.622,20.029,9.835,15.457,0.108,5.286,2.419,4.388,8.023
4,2025,16.067,2.629,7.771,11.704,10.437,62.579,13.794,46.512,74.325,...,22.043,12.418,18.703,9.075,14.502,1.613,4.874,1.550,6.724,7.520


In [30]:

# Validate Pfizer master dataset

print("Dataset shape:", pfizer_master.shape)

print("\nMissing values:")
print(pfizer_master.isnull().sum())

print("\nDuplicate years:", pfizer_master["Year"].duplicated().sum())

print("\nFinancial years:")
print(pfizer_master["Year"].tolist())


Dataset shape: (5, 22)

Missing values:
Year                       0
COGS                       0
Capital_Expenditure        0
Net_Income                 0
Operating_Cash_Flow        0
R&D                        0
Revenue                    0
SG&A                       0
Gross_Profit               0
Gross_Margin_%             0
Revenue_Growth_%           1
R&D_Intensity_%            0
SG&A_Ratio_%               0
Net_Margin_%               0
OCF_Margin_%               0
Free_Cash_Flow             0
FCF_Margin_%               0
Acquired_IPR&D             0
Amortization               0
Restructuring              0
Other_Income_Deductions    0
Income_Before_Tax          0
dtype: int64

Duplicate years: 0

Financial years:
[2021, 2022, 2023, 2024, 2025]


In [31]:

# Save Pfizer five-year master dataset

pfizer_output_path = Path("../data/processed/pfizer_financial_master.csv")

pfizer_output_path.parent.mkdir(parents=True, exist_ok=True)

pfizer_master.to_csv(pfizer_output_path, index=False)

print("Saved successfully:", pfizer_output_path)


Saved successfully: ..\data\processed\pfizer_financial_master.csv


In [32]:

# Locate product revenue tables in both Pfizer annual reports

for filing_name, soup in [
    ("FY2025", pfizer_soup),
    ("FY2024", pfizer_2024_soup)
]:
    print(f"\n--- {filing_name} filing ---")

    for i, table in enumerate(soup.find_all("table")):
        text = table.get_text(" ", strip=True).lower()

        if (
            "significant product revenues" in text
            or (
                "total revenues" in text
                and "primary care" in text
                and "comirnaty" in text
            )
        ):
            print("Table index:", i)
            print(text[:1800])



--- FY2025 filing ---
Table index: 279
(millions) year ended december 31, product primary indication or class 2025 2024 2023 total revenues $ 62,579 $ 63,627 $ 59,553 global biopharmaceuticals business (biopharma) $ 61,199 $ 62,400 $ 58,237 primary care $ 26,820 $ 30,135 $ 30,799 eliquis (a) nonvalvular atrial fibrillation, deep vein thrombosis, pulmonary embolism 7,961 7,366 6,747 prevnar family active immunization to prevent pneumonia, invasive disease and otitis media caused by streptococcus pneumoniae 6,494 6,411 6,501 comirnaty active immunization to prevent covid-19 4,367 5,353 11,220 paxlovid (b) covid-19 in certain high-risk patients 2,362 5,716 1,279 nurtec odt/vydura acute treatment of migraine and prevention of episodic migraine 1,424 1,263 928 abrysvo active immunization to prevent rsv infection 1,033 755 890 fsme-immun/ticovac active immunization to prevent tick-borne encephalitis disease 319 280 268 all other primary care various 2,860 2,991 2,968 specialty care $ 17,546

In [33]:

# Extract Pfizer product revenue table from FY2025 and FY2024 filings

def extract_pfizer_revenue_table(soup, table_index):
    table = soup.find_all("table")[table_index]

    records = []

    for row in table.find_all("tr"):
        cells = row.find_all(["th", "td"])

        values = [
            cell.get_text(" ", strip=True)
            for cell in cells
        ]

        if values:
            records.append(values)

    return records


pfizer_2025_revenue_rows = extract_pfizer_revenue_table(
    pfizer_soup, 279
)

pfizer_2024_revenue_rows = extract_pfizer_revenue_table(
    pfizer_2024_soup, 279
)

print("FY2025 filing rows:", len(pfizer_2025_revenue_rows))
print("FY2024 filing rows:", len(pfizer_2024_revenue_rows))

print("\nFY2025 table:")
for row in pfizer_2025_revenue_rows:
    print(row)


FY2025 filing rows: 39
FY2024 filing rows: 26

FY2025 table:
['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
['(MILLIONS)', '', '', '', 'Year Ended December 31,']
['PRODUCT', '', 'PRIMARY INDICATION OR CLASS', '', '2025', '', '2024', '', '2023']
['', '', '', '', '', '', '']
['TOTAL REVENUES', '', '', '', '$', '62,579', '', '', '$', '63,627', '', '', '$', '59,553', '']
['GLOBAL BIOPHARMACEUTICALS BUSINESS (BIOPHARMA)', '', '$', '61,199', '', '', '$', '62,400', '', '', '$', '58,237', '']
['Primary Care', '', '$', '26,820', '', '', '$', '30,135', '', '', '$', '30,799', '']
['Eliquis (a)', '', 'Nonvalvular atrial fibrillation, deep vein thrombosis, pulmonary embolism', '', '7,961', '', '', '7,366', '', '', '6,747', '']
['Prevnar family', '', 'Active immunization to prevent pneumonia, invasive disease and otitis media caused by Streptococcus pneumoniae', '', '6,494', '', '', '6,411', '', '', '6,501', '']
['Comirnaty', '', 'Active 

In [34]:

import re

# Convert the original FY2025 SEC table into structured records

revenue_records = []

for row in pfizer_2025_revenue_rows:

    if not row:
        continue

    product_name = row[0].strip()

    # Extract numeric values, ignoring currency symbols and blank cells
    numbers = []

    for cell in row[1:]:
        value = cell.strip().replace(",", "")

        if re.fullmatch(r"\d+", value):
            numbers.append(int(value))

    # FY2025 table contains 2025, 2024 and 2023
    if product_name and len(numbers) == 3:
        revenue_records.append({
            "Product_or_Category": product_name,
            "FY2025_USD_M": numbers[0],
            "FY2024_USD_M": numbers[1],
            "FY2023_USD_M": numbers[2]
        })

pfizer_revenue_3yr = pd.DataFrame(revenue_records)

display(pfizer_revenue_3yr)
print("Extracted records:", len(pfizer_revenue_3yr))


,Product_or_Category,FY2025_USD_M,FY2024_USD_M,FY2023_USD_M
0,PRODUCT,2025,2024,2023
1,TOTAL REVENUES,62579,63627,59553
2,GLOBAL BIOPHARMACEUTICALS BUSINESS (BIOPHARMA),61199,62400,58237
3,Primary Care,26820,30135,30799
4,Eliquis (a),7961,7366,6747
5,Prevnar family,6494,6411,6501
6,Comirnaty,4367,5353,11220
7,Paxlovid (b),2362,5716,1279
8,Nurtec ODT/Vydura,1424,1263,928
9,Abrysvo,1033,755,890


Extracted records: 36


In [35]:

# Remove header rows and clean the extracted revenue table

pfizer_revenue_3yr = pfizer_revenue_3yr[
    ~pfizer_revenue_3yr["Product_or_Category"].isin(
        ["PRODUCT", "TOTAL REVENUES"]
    )
].copy()

# Convert revenue columns to numeric
revenue_columns = [
    "FY2025_USD_M",
    "FY2024_USD_M",
    "FY2023_USD_M"
]

pfizer_revenue_3yr[revenue_columns] = (
    pfizer_revenue_3yr[revenue_columns].apply(
        pd.to_numeric, errors="coerce"
    )
)

pfizer_revenue_3yr = pfizer_revenue_3yr.reset_index(drop=True)

display(pfizer_revenue_3yr)

print("Clean records:", len(pfizer_revenue_3yr))


,Product_or_Category,FY2025_USD_M,FY2024_USD_M,FY2023_USD_M
0,GLOBAL BIOPHARMACEUTICALS BUSINESS (BIOPHARMA),61199,62400,58237
1,Primary Care,26820,30135,30799
2,Eliquis (a),7961,7366,6747
3,Prevnar family,6494,6411,6501
4,Comirnaty,4367,5353,11220
5,Paxlovid (b),2362,5716,1279
6,Nurtec ODT/Vydura,1424,1263,928
7,Abrysvo,1033,755,890
8,FSME-IMMUN/TicoVac,319,280,268
9,All other Primary Care,2860,2991,2968


Clean records: 34


In [36]:

# Inspect FY2024 filing product revenue table

for row in pfizer_2024_revenue_rows:
    print(row)


['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
['(MILLIONS)', '', '', '', 'Year Ended December 31,']
['PRODUCT', '', 'PRIMARY INDICATION OR CLASS', '', '2024', '', '2023', '', '2022']
['', '', '', '', '', '', '']
['TOTAL REVENUES', '', '', '', '$', '63,627', '', '', '$', '59,553', '', '', '$', '101,175', '']
['GLOBAL BIOPHARMACEUTICALS BUSINESS (BIOPHARMA)', '', '$', '62,400', '', '', '$', '58,237', '', '', '$', '99,826', '']
['Primary Care', '', '$', '30,135', '', '', '$', '30,799', '', '', '$', '73,181', '']
['Eliquis (a)', '', 'Nonvalvular atrial fibrillation, deep vein thrombosis, pulmonary embolism', '', '7,366', '', '', '6,747', '', '', '6,480', '']
['Prevnar family', '', 'Active immunization to prevent pneumonia, invasive disease and otitis media caused by Streptococcus pneumoniae', '', '6,411', '', '', '6,501', '', '', '6,342', '']
['Paxlovid (b)', '', 'COVID-19 in certain high-risk patients', '', '5,716', '', '', '1

In [37]:

# Extract product/category revenues from FY2024 filing

revenue_records_2024 = []

for row in pfizer_2024_revenue_rows:

    if not row:
        continue

    product_name = row[0].strip()

    if not product_name or product_name in [
        "PRODUCT", "TOTAL REVENUES"
    ]:
        continue

    values = []

    for cell in row[1:]:
        value = cell.strip().replace(",", "")

        if re.fullmatch(r"\d+", value):
            values.append(int(value))
        elif value == "—":
            values.append(None)

    if len(values) == 3:
        revenue_records_2024.append({
            "Product_or_Category": product_name,
            "FY2024_USD_M": values[0],
            "FY2023_USD_M": values[1],
            "FY2022_USD_M": values[2]
        })

pfizer_revenue_2024 = pd.DataFrame(revenue_records_2024)

display(pfizer_revenue_2024)
print("Extracted records:", len(pfizer_revenue_2024))


,Product_or_Category,FY2024_USD_M,FY2023_USD_M,FY2022_USD_M
0,GLOBAL BIOPHARMACEUTICALS BUSINESS (BIOPHARMA),62400,58237,99826.0
1,Primary Care,30135,30799,73181.0
2,Eliquis (a),7366,6747,6480.0
3,Prevnar family,6411,6501,6342.0
4,Paxlovid (b),5716,1279,18933.0
5,Comirnaty,5353,11220,37809.0
6,Nurtec ODT/Vydura,1263,928,213.0
7,Abrysvo,755,890,NaN
8,Premarin family,380,397,455.0
9,BMP2,352,338,277.0


Extracted records: 21


In [38]:

# Find continuation tables in Pfizer FY2024 10-K

for i in range(279, min(300, len(pfizer_2024_tables))):
    text = pfizer_2024_tables[i].get_text(" ", strip=True)

    if text.strip():
        print(f"\nTABLE {i}:")
        print(text[:1200])



TABLE 279:
(MILLIONS) Year Ended December 31, PRODUCT PRIMARY INDICATION OR CLASS 2024 2023 2022 TOTAL REVENUES $ 63,627 $ 59,553 $ 101,175 GLOBAL BIOPHARMACEUTICALS BUSINESS (BIOPHARMA) $ 62,400 $ 58,237 $ 99,826 Primary Care $ 30,135 $ 30,799 $ 73,181 Eliquis (a) Nonvalvular atrial fibrillation, deep vein thrombosis, pulmonary embolism 7,366 6,747 6,480 Prevnar family Active immunization to prevent pneumonia, invasive disease and otitis media caused by Streptococcus pneumoniae 6,411 6,501 6,342 Paxlovid (b) COVID-19 in certain high-risk patients 5,716 1,279 18,933 Comirnaty Active immunization to prevent COVID-19 5,353 11,220 37,809 Nurtec ODT/Vydura Acute treatment of migraine and prevention of episodic migraine 1,263 928 213 Abrysvo Active immunization to prevent RSV infection 755 890 — Premarin family Symptoms of menopause 380 397 455 BMP2 Bone graft for spinal fusion 352 338 277 FSME-IMMUN/TicoVac Active immunization to prevent tick-borne encephalitis disease 280 268 200 All oth

In [39]:

# Display the complete FY2024 continuation table

pfizer_2024_continuation = pfizer_2024_tables[282]

print(
    pfizer_2024_continuation.get_text(
        " ", strip=True
    )
)


(MILLIONS) Year Ended December 31, PRODUCT PRIMARY INDICATION OR CLASS 2024 2023 2022 Genotropin Replacement of human growth hormone 470 539 360 BeneFIX Hemophilia B 381 424 425 Cibinqo Atopic dermatitis 215 128 27 Oxbryta (d) Sickle cell disease 201 328 73 All other Hospital (e) Various 4,448 4,514 4,730 All other Specialty Care Various 907 792 743 Oncology $ 15,612 $ 12,450 $ 12,794 Ibrance HR-positive/HER2-negative metastatic breast cancer 4,367 4,753 5,120 Xtandi (f) mCRPC, nmCRPC, mCSPC, nmCSPC 2,039 1,659 1,650 Padcev Locally advanced or metastatic urothelial cancer 1,588 53 — Adcetris Hodgkin lymphoma and certain T-cell lymphomas 1,089 56 — Oncology biosimilars (g) Various 1,037 1,407 1,753 Inlyta Advanced RCC 978 1,036 1,003 Lorbrena ALK-positive metastatic NSCLC 731 539 343 Bosulif Philadelphia chromosome–positive chronic myelogenous leukemia 645 645 575 Braftovi/Mektovi Metastatic melanoma in patients with a BRAFV600E/K mutation and for metastatic NSCLC in patients with a BRA

In [40]:

# Extract complete Pfizer FY2024 product schedule

def parse_pfizer_product_rows(tables, table_indices, years):
    records = []
    current_category = None

    for table_index in table_indices:
        table = tables[table_index]

        for tr in table.find_all("tr"):
            cells = [
                cell.get_text(" ", strip=True).strip()
                for cell in tr.find_all(["th", "td"], recursive=False)
            ]

            cells = [c for c in cells if c]

            if not cells:
                continue

            name = cells[0].upper()

            # Identify category sections
            if name in ["PRIMARY CARE", "SPECIALTY CARE", "ONCOLOGY"]:
                current_category = name.title()
                
                # Extract category revenue totals
                nums = [
                    int(c.replace(",", ""))
                    for c in cells[1:]
                    if re.fullmatch(r"[\d,]+", c)
                ]

                if len(nums) >= 3:
                    records.append({
                        "Category": current_category,
                        "Product_or_Category": current_category,
                        **dict(zip(years, nums[-3:]))
                    })
                continue

            # Skip headings and division-level summaries
            if (
                name in [
                    "PRODUCT", "TOTAL REVENUES",
                    "GLOBAL BIOPHARMACEUTICALS BUSINESS (BIOPHARMA)",
                    "PFIZER CENTREONE (I)", "PFIZER IGNITE",
                    "BIOPHARMA",
                    "PFIZER U.S. COMMERCIAL DIVISION (U.S. PRIMARY CARE AND U.S. SPECIALTY CARE)",
                    "PFIZER ONCOLOGY DIVISION",
                    "PFIZER INTERNATIONAL COMMERCIAL DIVISION"
                ]
                or current_category is None
            ):
                continue

            # Read only cells containing complete numeric values
            nums = []

            for c in cells[1:]:
                if re.fullmatch(r"[\d,]+", c):
                    nums.append(int(c.replace(",", "")))
                elif c == "—":
                    nums.append(None)

            if len(nums) >= 3:
                records.append({
                    "Category": current_category,
                    "Product_or_Category": cells[0],
                    **dict(zip(years, nums[-3:]))
                })

    return pd.DataFrame(records)


pfizer_2024_complete = parse_pfizer_product_rows(
    pfizer_2024_tables,
    [279, 282],
    ["FY2024_USD_M", "FY2023_USD_M", "FY2022_USD_M"]
)

display(pfizer_2024_complete)

print("Total extracted records:", len(pfizer_2024_complete))
print("\nRecords by category:")
print(pfizer_2024_complete.groupby("Category").size())


,Category,Product_or_Category,FY2024_USD_M,FY2023_USD_M,FY2022_USD_M
0,Primary Care,Primary Care,30135,30799,73181.0
1,Primary Care,Eliquis (a),7366,6747,6480.0
2,Primary Care,Prevnar family,6411,6501,6342.0
3,Primary Care,Paxlovid (b),5716,1279,18933.0
4,Primary Care,Comirnaty,5353,11220,37809.0
5,Primary Care,Nurtec ODT/Vydura,1263,928,213.0
6,Primary Care,Abrysvo,755,890,NaN
7,Primary Care,Premarin family,380,397,455.0
8,Primary Care,BMP2,352,338,277.0
9,Primary Care,FSME-IMMUN/TicoVac,280,268,200.0


Total extracted records: 43

Records by category:
Category
Oncology          17
Primary Care      11
Specialty Care    15
dtype: int64


In [41]:
# Remove category-total rows and alliance/royalty disclosures
exclude_rows = [
    "Primary Care",
    "Specialty Care",
    "Oncology",
    "Total Alliance revenues included above",
    "Total Royalty revenues included above"
]

pfizer_products_2024 = pfizer_2024_complete[
    ~pfizer_2024_complete["Product_or_Category"].isin(exclude_rows)
].copy()

# Reset index
pfizer_products_2024.reset_index(drop=True, inplace=True)

# Check the cleaned dataset
print("Product records:", len(pfizer_products_2024))
print("\nRecords by category:")
print(pfizer_products_2024["Category"].value_counts())

display(pfizer_products_2024.head())

Product records: 38

Records by category:
Category
Specialty Care    14
Oncology          14
Primary Care      10
Name: count, dtype: int64


,Category,Product_or_Category,FY2024_USD_M,FY2023_USD_M,FY2022_USD_M
0,Primary Care,Eliquis (a),7366,6747,6480.0
1,Primary Care,Prevnar family,6411,6501,6342.0
2,Primary Care,Paxlovid (b),5716,1279,18933.0
3,Primary Care,Comirnaty,5353,11220,37809.0
4,Primary Care,Nurtec ODT/Vydura,1263,928,213.0


In [42]:
# Calculate product revenue totals by portfolio
portfolio_check = (
    pfizer_products_2024
    .groupby("Category")[
        ["FY2024_USD_M", "FY2023_USD_M", "FY2022_USD_M"]
    ]
    .sum()
)

display(portfolio_check)

,FY2024_USD_M,FY2023_USD_M,FY2022_USD_M
Category,,,
Oncology,15612,12450,12794.0
Primary Care,30135,30801,73182.0
Specialty Care,16652,14988,13851.0


In [43]:
pfizer_products_2024[
    pfizer_products_2024["Category"] == "Primary Care"
][
    ["Product_or_Category", "FY2024_USD_M",
     "FY2023_USD_M", "FY2022_USD_M"]
]

,Product_or_Category,FY2024_USD_M,FY2023_USD_M,FY2022_USD_M
0,Eliquis (a),7366,6747,6480.0
1,Prevnar family,6411,6501,6342.0
2,Paxlovid (b),5716,1279,18933.0
3,Comirnaty,5353,11220,37809.0
4,Nurtec ODT/Vydura,1263,928,213.0
5,Abrysvo,755,890,NaN
6,Premarin family,380,397,455.0
7,BMP2,352,338,277.0
8,FSME-IMMUN/TicoVac,280,268,200.0
9,All other Primary Care,2259,2233,2473.0


In [44]:
pfizer_2025_tables = pfizer_soup.find_all("table")

for i in range(279, min(300, len(pfizer_2025_tables))):
    table_text = pfizer_2025_tables[i].get_text(" ", strip=True)

    if (
        "All other Oncology" in table_text
        or "PFIZER CENTREONE" in table_text
        or "Tukysa" in table_text
    ):
        print(f"\nTABLE INDEX: {i}")
        print(table_text[:5000])


TABLE INDEX: 282
(MILLIONS) Year Ended December 31, PRODUCT PRIMARY INDICATION OR CLASS 2025 2024 2023 Tukysa Unresectable or metastatic HER2-positive breast cancer; RAS wild-type, HER2-positive unresectable or metastatic colorectal cancer 463 480 18 Aromasin Post-menopausal early and advanced breast cancer 450 347 301 Orgovyx (g) Advanced prostate cancer 421 201 120 Elrexfio Relapsed or refractory multiple myeloma 304 133 10 Talzenna Treatment of BRCA gene-mutated, HER2-negative, inoperable or recurrent breast cancer; and, in combination with Xtandi (enzalutamide), of adult patients with HRR gene-mutated mCRPC 182 117 64 Tivdak Recurrent or mCC with disease progression on or after chemotherapy 147 131 4 All other Oncology Various 1,127 1,122 1,308 PFIZER CENTREONE (h) $ 1,338 $ 1,146 $ 1,272 PFIZER IGNITE $ 41 $ 82 $ 44 BIOPHARMA $ 61,199 $ 62,400 $ 58,237 PFIZER U.S. COMMERCIAL DIVISION (i) 36,708 38,332 27,749 PFIZER INTERNATIONAL COMMERCIAL DIVISION 24,491 24,068 30,488 Total Alli

In [45]:
pfizer_2025_complete = parse_pfizer_product_rows(
    pfizer_2025_tables,
    [279, 282],
    ["FY2025_USD_M", "FY2024_USD_M", "FY2023_USD_M"]
)

print("Extracted records:", len(pfizer_2025_complete))

print("\nRecords by category:")
print(pfizer_2025_complete["Category"].value_counts())

display(pfizer_2025_complete.tail(15))


Extracted records: 44

Records by category:
Category
Oncology          21
Specialty Care    14
Primary Care       9
Name: count, dtype: int64


,Category,Product_or_Category,FY2025_USD_M,FY2024_USD_M,FY2023_USD_M
29,Oncology,Inlyta,923,978,1036
30,Oncology,Adcetris (e),907,1089,56
31,Oncology,Braftovi/Mektovi,716,607,477
32,Oncology,Bosulif,611,645,645
33,Oncology,Tukysa,463,480,18
34,Oncology,Aromasin,450,347,301
35,Oncology,Orgovyx (g),421,201,120
36,Oncology,Elrexfio,304,133,10
37,Oncology,Talzenna,182,117,64
38,Oncology,Tivdak,147,131,4


In [46]:
# Keep portfolio totals separately
portfolio_names = ["Primary Care", "Specialty Care", "Oncology"]

pfizer_portfolio_2025 = pfizer_2025_complete[
    pfizer_2025_complete["Product_or_Category"].isin(portfolio_names)
].copy()

# Remove non-product and supplementary disclosures
exclude_patterns = (
    "PFIZER",
    "TOTAL ALLIANCE",
    "TOTAL ROYALTY",
    "BIOPHARMA"
)

pfizer_products_2025 = pfizer_2025_complete[
    ~pfizer_2025_complete["Product_or_Category"].isin(portfolio_names)
    &
    ~pfizer_2025_complete["Product_or_Category"].str.upper().str.startswith(exclude_patterns)
].copy()

pfizer_products_2025.reset_index(drop=True, inplace=True)

print("Product records:", len(pfizer_products_2025))
print("\nProducts by portfolio:")
print(pfizer_products_2025["Category"].value_counts())

print("\nPortfolio totals:")
display(pfizer_portfolio_2025)

print("\nLast 10 product records:")
display(pfizer_products_2025.tail(10))

Product records: 37

Products by portfolio:
Category
Oncology          16
Specialty Care    13
Primary Care       8
Name: count, dtype: int64

Portfolio totals:


,Category,Product_or_Category,FY2025_USD_M,FY2024_USD_M,FY2023_USD_M
0,Primary Care,Primary Care,26820,30135,30799
9,Specialty Care,Specialty Care,17546,16652,14988
23,Oncology,Oncology,16834,15612,12450



Last 10 product records:


,Category,Product_or_Category,FY2025_USD_M,FY2024_USD_M,FY2023_USD_M
27,Oncology,Adcetris (e),907,1089,56
28,Oncology,Braftovi/Mektovi,716,607,477
29,Oncology,Bosulif,611,645,645
30,Oncology,Tukysa,463,480,18
31,Oncology,Aromasin,450,347,301
32,Oncology,Orgovyx (g),421,201,120
33,Oncology,Elrexfio,304,133,10
34,Oncology,Talzenna,182,117,64
35,Oncology,Tivdak,147,131,4
36,Oncology,All other Oncology,1127,1122,1308


In [47]:
# Sum product revenue by portfolio
product_sum_2025 = (
    pfizer_products_2025
    .groupby("Category")[
        ["FY2025_USD_M", "FY2024_USD_M", "FY2023_USD_M"]
    ]
    .sum()
)

# Compare against reported portfolio totals
reported_sum_2025 = (
    pfizer_portfolio_2025
    .set_index("Category")[
        ["FY2025_USD_M", "FY2024_USD_M", "FY2023_USD_M"]
    ]
)

# Calculate reconciliation differences
reconciliation_2025 = product_sum_2025 - reported_sum_2025

print("Product revenue totals:")
display(product_sum_2025)

print("Reconciliation differences (product sum - reported total):")
display(reconciliation_2025)

Product revenue totals:


,FY2025_USD_M,FY2024_USD_M,FY2023_USD_M
Category,,,
Oncology,16831,15612,12450
Primary Care,26820,30135,30801
Specialty Care,17545,16652,14989


Reconciliation differences (product sum - reported total):


,FY2025_USD_M,FY2024_USD_M,FY2023_USD_M
Category,,,
Oncology,-3,0,0
Primary Care,0,0,2
Specialty Care,-1,0,1


In [48]:
pfizer_2023_url = (
    "https://www.sec.gov/Archives/edgar/data/78003/"
    "000007800324000039/pfe-20231231.htm"
)

pfizer_2023_response = requests.get(
    pfizer_2023_url,
    headers=headers
)

print("Status code:", pfizer_2023_response.status_code)
print("Downloaded size:", len(pfizer_2023_response.content))

pfizer_2023_soup = BeautifulSoup(
    pfizer_2023_response.content,
    "html.parser"
)

pfizer_2023_tables = pfizer_2023_soup.find_all("table")

print("Number of tables:", len(pfizer_2023_tables))

Status code: 200
Downloaded size: 5349289
Number of tables: 316


In [49]:
for i, table in enumerate(pfizer_2023_tables):
    table_text = table.get_text(" ", strip=True)

    if (
        "GLOBAL BIOPHARMACEUTICALS BUSINESS" in table_text
        and "Primary Care" in table_text
        and "Comirnaty" in table_text
        and "2021" in table_text
    ):
        print(f"\nTABLE INDEX: {i}")
        print(table_text[:2500])


TABLE INDEX: 279
(MILLIONS) Year Ended December 31, PRODUCT PRIMARY INDICATION OR CLASS 2023 2022 2021 TOTAL REVENUES $ 58,496 $ 100,330 $ 81,288 GLOBAL BIOPHARMACEUTICALS BUSINESS (BIOPHARMA) $ 57,186 $ 98,988 $ 79,557 Primary Care $ 30,589 $ 73,023 $ 52,029 Comirnaty direct sales and alliance revenues (a) Active immunization to prevent COVID-19 11,220 37,806 36,781 Eliquis alliance revenues and direct sales Nonvalvular atrial fibrillation, deep vein thrombosis, pulmonary embolism 6,747 6,480 5,970 Prevnar family Active immunization to prevent pneumonia, invasive disease and otitis media caused by Streptococcus pneumoniae 6,440 6,337 5,272 Paxlovid (b) COVID-19 in certain high-risk patients 1,279 18,933 76 Nurtec ODT/Vydura Acute treatment of migraine and prevention of episodic migraine 928 213 — Abrysvo Active immunization to prevent RSV infection 890 — — Premarin family Symptoms of menopause 397 455 563 BMP2 Bone graft for spinal fusion 338 277 266 FSME-IMMUN/TicoVac Active immuniz

In [50]:
for i in range(280, min(300, len(pfizer_2023_tables))):
    table_text = pfizer_2023_tables[i].get_text(" ", strip=True)

    if (
        "All other Oncology" in table_text
        or "PFIZER CENTREONE" in table_text
        or "Tukysa" in table_text
    ):
        print(f"\nTABLE INDEX: {i}")
        print(table_text[:5000])


TABLE INDEX: 282
(MILLIONS) Year Ended December 31, PRODUCT PRIMARY INDICATION OR CLASS 2023 2022 2021 Ruxience Non-hodgkin’s lymphoma, chronic lymphocytic leukemia, granulomatosis with polyangiitis (Wegener’s Granulomatosis) and microscopic polyangiitis 390 458 491 Xalkori ALK-positive and Proto-Oncogene 1, Receptor Tyrosine Kinase-positive advanced NSCLC 374 465 493 Retacrit Anemia 340 394 444 Aromasin Post-menopausal early and advanced breast cancer 301 248 211 Besponsa Relapsed or refractory B-cell acute lymphoblastic leukemia 236 219 192 Braftovi In combination with Mektovi for metastatic melanoma in patients with a BRAF V600E/K mutation and for metastatic NSCLC in patients with a BRAF V600E mutation; and In combination with Erbitux (cetuximab) (d) for the treatment of BRAF V600E -mutant mCRC after prior therapy 213 194 187 Bavencio alliance revenues (e) Locally advanced or metastatic urothelial carcinoma; metastatic Merkel cell carcinoma; immunotherapy and tyrosine kinase inhibi

In [51]:
pfizer_2023_complete = parse_pfizer_product_rows(
    pfizer_2023_tables,
    [279, 282],
    ["FY2023_USD_M", "FY2022_USD_M", "FY2021_USD_M"]
)

print("Extracted records:", len(pfizer_2023_complete))

print("\nRecords by category:")
print(pfizer_2023_complete["Category"].value_counts())

display(pfizer_2023_complete.tail(15))

Extracted records: 60

Records by category:
Category
Oncology          25
Specialty Care    22
Primary Care      13
Name: count, dtype: int64


,Category,Product_or_Category,FY2023_USD_M,FY2022_USD_M,FY2021_USD_M
45,Oncology,Aromasin,301,248.0,211.0
46,Oncology,Besponsa,236,219.0,192.0
47,Oncology,Braftovi,213,194.0,187.0
48,Oncology,Bavencio alliance revenues (e),190,271.0,178.0
49,Oncology,Sutent,180,347.0,673.0
50,Oncology,Mektovi,174,176.0,155.0
51,Oncology,Trazimera,91,203.0,197.0
52,Oncology,Padcev (f),52,NaN,NaN
53,Oncology,Adcetris (f),46,NaN,NaN
54,Oncology,Tukysa (f),17,NaN,NaN


In [52]:
# Keep portfolio totals separately
pfizer_portfolio_2023 = pfizer_2023_complete[
    pfizer_2023_complete["Product_or_Category"].isin(
        ["Primary Care", "Specialty Care", "Oncology"]
    )
].copy()

# Exclude business-level and supplementary disclosures
exclude_names = [
    "BUSINESS INNOVATION (g)",
    "Pfizer CentreOne (h)",
    "Pfizer Ignite",
    "Total Alliance revenues included above",
    "Total Royalty revenues included above"
]

pfizer_products_2023 = pfizer_2023_complete[
    ~pfizer_2023_complete["Product_or_Category"].isin(
        ["Primary Care", "Specialty Care", "Oncology"] + exclude_names
    )
].copy()

pfizer_products_2023.reset_index(drop=True, inplace=True)

print("Product records:", len(pfizer_products_2023))
print("\nRecords by portfolio:")
print(pfizer_products_2023["Category"].value_counts())

display(pfizer_products_2023.tail(12))

Product records: 54

Records by portfolio:
Category
Specialty Care    21
Oncology          21
Primary Care      12
Name: count, dtype: int64


,Category,Product_or_Category,FY2023_USD_M,FY2022_USD_M,FY2021_USD_M
42,Oncology,Aromasin,301,248.0,211.0
43,Oncology,Besponsa,236,219.0,192.0
44,Oncology,Braftovi,213,194.0,187.0
45,Oncology,Bavencio alliance revenues (e),190,271.0,178.0
46,Oncology,Sutent,180,347.0,673.0
47,Oncology,Mektovi,174,176.0,155.0
48,Oncology,Trazimera,91,203.0,197.0
49,Oncology,Padcev (f),52,NaN,NaN
50,Oncology,Adcetris (f),46,NaN,NaN
51,Oncology,Tukysa (f),17,NaN,NaN


In [53]:
# Start with the FY2025 product dataset
pfizer_product_master = pfizer_products_2025[
    ["Category", "Product_or_Category", "FY2025_USD_M",
     "FY2024_USD_M", "FY2023_USD_M"]
].copy()

# Add FY2022 from the FY2024 filing
fy2022_data = pfizer_products_2024[
    ["Category", "Product_or_Category", "FY2022_USD_M"]
].copy()

pfizer_product_master = pfizer_product_master.merge(
    fy2022_data,
    on=["Category", "Product_or_Category"],
    how="outer"
)

# Add FY2021 from the FY2023 filing
fy2021_data = pfizer_products_2023[
    ["Category", "Product_or_Category", "FY2021_USD_M"]
].copy()

pfizer_product_master = pfizer_product_master.merge(
    fy2021_data,
    on=["Category", "Product_or_Category"],
    how="outer"
)

# Arrange columns
pfizer_product_master = pfizer_product_master[
    [
        "Category",
        "Product_or_Category",
        "FY2021_USD_M",
        "FY2022_USD_M",
        "FY2023_USD_M",
        "FY2024_USD_M",
        "FY2025_USD_M"
    ]
].sort_values(
    ["Category", "Product_or_Category"]
).reset_index(drop=True)

print("Master dataset shape:", pfizer_product_master.shape)
print("\nRecords by category:")
print(pfizer_product_master["Category"].value_counts())

display(pfizer_product_master.head(15))

Master dataset shape: (76, 7)

Records by category:
Category
Oncology          34
Specialty Care    28
Primary Care      14
Name: count, dtype: int64


,Category,Product_or_Category,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
0,Oncology,Adcetris,NaN,NaN,NaN,NaN,NaN
1,Oncology,Adcetris (e),NaN,NaN,56.0,1089.0,907.0
2,Oncology,Adcetris (f),NaN,NaN,NaN,NaN,NaN
3,Oncology,All other Oncology,238.0,1846.0,1308.0,1122.0,1127.0
4,Oncology,Aromasin,211.0,NaN,301.0,347.0,450.0
5,Oncology,Bavencio alliance revenues (e),178.0,NaN,NaN,NaN,NaN
6,Oncology,Besponsa,192.0,NaN,NaN,NaN,NaN
7,Oncology,Bosulif,540.0,575.0,645.0,645.0,611.0
8,Oncology,Braftovi,187.0,NaN,NaN,NaN,NaN
9,Oncology,Braftovi/Mektovi,NaN,456.0,477.0,607.0,716.0


In [54]:
# 1. Find rows with no revenue values in any year
year_columns = [
    "FY2021_USD_M",
    "FY2022_USD_M",
    "FY2023_USD_M",
    "FY2024_USD_M",
    "FY2025_USD_M"
]

empty_rows = pfizer_product_master[
    pfizer_product_master[year_columns].isna().all(axis=1)
]

print("Rows with no revenue values:", len(empty_rows))
display(empty_rows)

# 2. Inspect products with potentially changed names
name_checks = [
    "Adcetris",
    "Braftovi",
    "Mektovi",
    "Aromasin",
    "Bosulif",
    "Padcev",
    "Tukysa"
]

for product in name_checks:
    print(f"\n--- {product} ---")
    display(
        pfizer_product_master[
            pfizer_product_master["Product_or_Category"]
            .str.contains(product, case=False, na=False)
        ]
    )

Rows with no revenue values: 6


,Category,Product_or_Category,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
0,Oncology,Adcetris,NaN,NaN,NaN,NaN,NaN
2,Oncology,Adcetris (f),NaN,NaN,NaN,NaN,NaN
19,Oncology,Padcev (f),NaN,NaN,NaN,NaN,NaN
25,Oncology,Tivdak (f),NaN,NaN,NaN,NaN,NaN
28,Oncology,Tukysa (f),NaN,NaN,NaN,NaN,NaN
64,Specialty Care,Oxbryta,NaN,NaN,NaN,NaN,NaN



--- Adcetris ---


,Category,Product_or_Category,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
0,Oncology,Adcetris,NaN,NaN,NaN,NaN,NaN
1,Oncology,Adcetris (e),NaN,NaN,56.0,1089.0,907.0
2,Oncology,Adcetris (f),NaN,NaN,NaN,NaN,NaN



--- Braftovi ---


,Category,Product_or_Category,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
8,Oncology,Braftovi,187.0,NaN,NaN,NaN,NaN
9,Oncology,Braftovi/Mektovi,NaN,456.0,477.0,607.0,716.0


... (6 outputs left out)


--- Padcev ---


,Category,Product_or_Category,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
18,Oncology,Padcev,NaN,NaN,53.0,1588.0,1940.0
19,Oncology,Padcev (f),NaN,NaN,NaN,NaN,NaN



--- Tukysa ---


,Category,Product_or_Category,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
27,Oncology,Tukysa,NaN,NaN,18.0,480.0,463.0
28,Oncology,Tukysa (f),NaN,NaN,NaN,NaN,NaN


In [55]:
import re
import pandas as pd

pfizer_harmonized = pfizer_product_master.copy()

# Remove footnote markers such as (a), (e), (f)
pfizer_harmonized["Product_or_Category"] = (
    pfizer_harmonized["Product_or_Category"]
    .str.replace(r"\s*\([a-z]\)$", "", regex=True)
    .str.strip()
)

# Combine historical Braftovi and Mektovi values
braftovi_rows = pfizer_harmonized[
    pfizer_harmonized["Product_or_Category"].isin(
        ["Braftovi", "Mektovi", "Braftovi/Mektovi"]
    )
]

year_columns = [
    "FY2021_USD_M",
    "FY2022_USD_M",
    "FY2023_USD_M",
    "FY2024_USD_M",
    "FY2025_USD_M"
]

braftovi_combined = braftovi_rows[year_columns].sum(
    min_count=1
)

# Remove the original three rows
pfizer_harmonized = pfizer_harmonized[
    ~pfizer_harmonized["Product_or_Category"].isin(
        ["Braftovi", "Mektovi", "Braftovi/Mektovi"]
    )
].copy()

# Add the combined product family
new_row = {
    "Category": "Oncology",
    "Product_or_Category": "Braftovi/Mektovi"
}

new_row.update(braftovi_combined.to_dict())

pfizer_harmonized = pd.concat(
    [pfizer_harmonized, pd.DataFrame([new_row])],
    ignore_index=True
)

# Remove rows with no revenue in any year
pfizer_harmonized = pfizer_harmonized.dropna(
    subset=year_columns,
    how="all"
)

pfizer_harmonized = pfizer_harmonized.sort_values(
    ["Category", "Product_or_Category"]
).reset_index(drop=True)

print("Harmonized dataset shape:", pfizer_harmonized.shape)
display(
    pfizer_harmonized[
        pfizer_harmonized["Product_or_Category"].str.contains(
            "Adcetris|Padcev|Tivdak|Tukysa|Braftovi|Oxbryta",
            case=False,
            na=False
        )
    ]
)

Harmonized dataset shape: (68, 7)


,Category,Product_or_Category,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
0,Oncology,Adcetris,NaN,NaN,56.0,1089.0,907.0
6,Oncology,Braftovi/Mektovi,342.0,456.0,477.0,607.0,716.0
14,Oncology,Padcev,NaN,NaN,53.0,1588.0,1940.0
19,Oncology,Tivdak,NaN,NaN,4.0,131.0,147.0
21,Oncology,Tukysa,NaN,NaN,18.0,480.0,463.0
57,Specialty Care,Oxbryta,NaN,73.0,NaN,NaN,NaN


In [56]:
# Identify duplicate product names after footnote removal
duplicate_products = pfizer_harmonized[
    pfizer_harmonized.duplicated(
        subset=["Category", "Product_or_Category"],
        keep=False
    )
].sort_values(["Category", "Product_or_Category"])

print("Duplicate product rows:", len(duplicate_products))
display(duplicate_products)

Duplicate product rows: 8


,Category,Product_or_Category,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
11,Oncology,Oncology biosimilars,NaN,NaN,1407.0,1037.0,1301.0
12,Oncology,Oncology biosimilars,NaN,1753.0,NaN,NaN,NaN
23,Oncology,Xtandi,NaN,NaN,1659.0,2039.0,2194.0
24,Oncology,Xtandi,NaN,1650.0,NaN,NaN,NaN
42,Specialty Care,All other Hospital,NaN,NaN,4514.0,4167.0,4030.0
43,Specialty Care,All other Hospital,NaN,4730.0,NaN,NaN,NaN
55,Specialty Care,Octagam,NaN,NaN,245.0,509.0,418.0
56,Specialty Care,Octagam,NaN,186.0,NaN,NaN,NaN


In [57]:
# Consolidate duplicate product names year by year
pfizer_harmonized_final = (
    pfizer_harmonized
    .groupby(
        ["Category", "Product_or_Category"],
        as_index=False
    )[year_columns]
    .sum(min_count=1)
)

# Sort the final dataset
pfizer_harmonized_final = pfizer_harmonized_final.sort_values(
    ["Category", "Product_or_Category"]
).reset_index(drop=True)

print("Final dataset shape:", pfizer_harmonized_final.shape)

print("\nDuplicate records remaining:")
print(
    pfizer_harmonized_final.duplicated(
        subset=["Category", "Product_or_Category"]
    ).sum()
)

display(
    pfizer_harmonized_final[
        pfizer_harmonized_final["Product_or_Category"].isin([
            "Oncology biosimilars",
            "Xtandi",
            "All other Hospital",
            "Octagam"
        ])
    ]
)

Final dataset shape: (64, 7)

Duplicate records remaining:
0


,Category,Product_or_Category,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
11,Oncology,Oncology biosimilars,NaN,1753.0,1407.0,1037.0,1301.0
22,Oncology,Xtandi,NaN,1650.0,1659.0,2039.0,2194.0
40,Specialty Care,All other Hospital,NaN,4730.0,4514.0,4167.0,4030.0
52,Specialty Care,Octagam,NaN,186.0,245.0,509.0,418.0


In [60]:
# Extract portfolio-level rows from FY2024 filing
pfizer_portfolio_2024 = pfizer_2024_complete[
    pfizer_2024_complete["Product_or_Category"].isin(
        ["Primary Care", "Specialty Care", "Oncology"]
    )
].copy()

# Extract portfolio-level rows from FY2025 filing
pfizer_portfolio_2025 = pfizer_2025_complete[
    pfizer_2025_complete["Product_or_Category"].isin(
        ["Primary Care", "Specialty Care", "Oncology"]
    )
].copy()

# Display all three filing extracts
print("FY2023 filing — portfolio totals")
display(pfizer_portfolio_2023)

print("FY2024 filing — portfolio totals")
display(pfizer_portfolio_2024)

print("FY2025 filing — portfolio totals")
display(pfizer_portfolio_2025)

FY2023 filing — portfolio totals


,Category,Product_or_Category,FY2023_USD_M,FY2022_USD_M,FY2021_USD_M
0,Primary Care,Primary Care,30589,73023.0,52029.0
13,Specialty Care,Specialty Care,14970,13833.0,15194.0
35,Oncology,Oncology,11627,12132.0,12333.0


FY2024 filing — portfolio totals


,Category,Product_or_Category,FY2024_USD_M,FY2023_USD_M,FY2022_USD_M
0,Primary Care,Primary Care,30135,30799,73181.0
11,Specialty Care,Specialty Care,16652,14988,13851.0
26,Oncology,Oncology,15612,12450,12794.0


FY2025 filing — portfolio totals


,Category,Product_or_Category,FY2025_USD_M,FY2024_USD_M,FY2023_USD_M
0,Primary Care,Primary Care,26820,30135,30799
9,Specialty Care,Specialty Care,17546,16652,14988
23,Oncology,Oncology,16834,15612,12450


In [61]:
# Extract category totals into a consistent format
def get_portfolio_values(df, year_columns):
    return (
        df.set_index("Category")[year_columns]
    )

# Select the latest available filing for each year
reported_totals = pd.DataFrame(index=[
    "Primary Care",
    "Specialty Care",
    "Oncology"
])

# FY2021 from FY2023 filing
reported_totals["FY2021_USD_M"] = (
    get_portfolio_values(
        pfizer_portfolio_2023, ["FY2021_USD_M"]
    )["FY2021_USD_M"]
)

# FY2022 from FY2024 filing
reported_totals["FY2022_USD_M"] = (
    get_portfolio_values(
        pfizer_portfolio_2024, ["FY2022_USD_M"]
    )["FY2022_USD_M"]
)

# FY2023–FY2025 from FY2025 filing
reported_totals[
    ["FY2023_USD_M", "FY2024_USD_M", "FY2025_USD_M"]
] = get_portfolio_values(
    pfizer_portfolio_2025,
    ["FY2023_USD_M", "FY2024_USD_M", "FY2025_USD_M"]
)

print("Five-year reported portfolio totals:")
display(reported_totals)

Five-year reported portfolio totals:


,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
Primary Care,52029.0,73181.0,30799,30135,26820
Specialty Care,15194.0,13851.0,14988,16652,17546
Oncology,12333.0,12794.0,12450,15612,16834


In [62]:
# Sum product-level revenue by portfolio
product_totals = (
    pfizer_harmonized_final
    .groupby("Category")[year_columns]
    .sum(min_count=1)
)

# Calculate reconciliation differences
reconciliation = product_totals - reported_totals

print("Product revenue totals:")
display(product_totals)

print("Reported portfolio totals:")
display(reported_totals)

print("Reconciliation differences:")
display(reconciliation)

Product revenue totals:


,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
Category,,,,,
Oncology,12333.0,12794.0,12450.0,15612.0,16831.0
Primary Care,52028.0,73182.0,30801.0,30135.0,26820.0
Specialty Care,15192.0,13851.0,14989.0,16652.0,17545.0


Reported portfolio totals:


,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
Primary Care,52029.0,73181.0,30799,30135,26820
Specialty Care,15194.0,13851.0,14988,16652,17546
Oncology,12333.0,12794.0,12450,15612,16834


Reconciliation differences:


,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
Category,,,,,
Oncology,0.0,0.0,0.0,0.0,-3.0
Primary Care,-1.0,1.0,2.0,0.0,0.0
Specialty Care,-2.0,0.0,1.0,0.0,-1.0


In [63]:
from pathlib import Path

# Create processed-data directory if it doesn't exist
output_folder = Path("../data/processed")
output_folder.mkdir(parents=True, exist_ok=True)

# Save five-year product dataset
pfizer_harmonized_final.to_csv(
    output_folder / "pfizer_product_revenue_2021_2025.csv",
    index=False
)

# Save reported portfolio totals
reported_totals.to_csv(
    output_folder / "pfizer_portfolio_revenue_2021_2025.csv"
)

# Save reconciliation report
reconciliation.to_csv(
    output_folder / "pfizer_revenue_reconciliation.csv"
)

print("Files saved successfully:")
for file in output_folder.glob("pfizer_*.csv"):
    print(file.name, "|", file.stat().st_size, "bytes")

Files saved successfully:
pfizer_financial_master.csv | 1439 bytes
pfizer_portfolio_revenue_2021_2025.csv | 209 bytes
pfizer_product_revenue_2021_2025.csv | 3152 bytes
pfizer_revenue_reconciliation.csv | 179 bytes


In [64]:
import pandas as pd

# Load the saved portfolio revenue data
pfizer_portfolio = pd.read_csv(
    "../data/processed/pfizer_portfolio_revenue_2021_2025.csv",
    index_col=0
)

# Display dataset
display(pfizer_portfolio)

print("Dataset shape:", pfizer_portfolio.shape)
print("Missing values:")
print(pfizer_portfolio.isnull().sum())

,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
Primary Care,52029.0,73181.0,30799,30135,26820
Specialty Care,15194.0,13851.0,14988,16652,17546
Oncology,12333.0,12794.0,12450,15612,16834


Dataset shape: (3, 5)
Missing values:
FY2021_USD_M    0
FY2022_USD_M    0
FY2023_USD_M    0
FY2024_USD_M    0
FY2025_USD_M    0
dtype: int64


In [65]:
# Calculate year-over-year revenue growth
pfizer_yoy_growth = pfizer_portfolio.pct_change(
    axis=1,
    fill_method=None
) * 100

# Display results
display(pfizer_yoy_growth.round(2))

,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
Primary Care,NaN,40.65,-57.91,-2.16,-11.00
Specialty Care,NaN,-8.84,8.21,11.10,5.37
Oncology,NaN,3.74,-2.69,25.40,7.83


In [66]:
# Calculate five-year revenue CAGR
pfizer_cagr = (
    (pfizer_portfolio["FY2025_USD_M"] /
     pfizer_portfolio["FY2021_USD_M"]) ** (1/4) - 1
) * 100

# Display CAGR
pfizer_cagr = pfizer_cagr.to_frame(
    name="FY2021_FY2025_CAGR_%"
)

display(pfizer_cagr.round(2))

,FY2021_FY2025_CAGR_%
Primary Care,-15.27
Specialty Care,3.66
Oncology,8.09


In [67]:
# Calculate portfolio contribution to combined portfolio revenue
pfizer_portfolio_share = (
    pfizer_portfolio.div(
        pfizer_portfolio.sum(axis=0),
        axis=1
    ) * 100
)

# Display contribution percentages
display(pfizer_portfolio_share.round(2))

,FY2021_USD_M,FY2022_USD_M,FY2023_USD_M,FY2024_USD_M,FY2025_USD_M
Primary Care,65.4,73.31,52.89,48.29,43.82
Specialty Care,19.1,13.88,25.74,26.69,28.67
Oncology,15.5,12.82,21.38,25.02,27.51


In [68]:
# Identify top 10 products by FY2025 revenue
pfizer_top_products = (
    pfizer_harmonized_final
    .sort_values("FY2025_USD_M", ascending=False)
    .head(10)
)

display(
    pfizer_top_products[
        ["Category", "Product_or_Category", "FY2025_USD_M"]
    ]
)

,Category,Product_or_Category,FY2025_USD_M
30,Primary Care,Eliquis,7961.0
37,Primary Care,Prevnar family,6494.0
59,Specialty Care,Vyndaqel family,6380.0
28,Primary Care,Comirnaty,4367.0
8,Oncology,Ibrance,4122.0
40,Specialty Care,All other Hospital,4030.0
26,Primary Care,All other Primary Care,2860.0
35,Primary Care,Paxlovid,2362.0
22,Oncology,Xtandi,2194.0
13,Oncology,Padcev,1940.0


In [69]:
# Calculate absolute revenue change from FY2021 to FY2025
pfizer_harmonized_final["Revenue_Change_USD_M"] = (
    pfizer_harmonized_final["FY2025_USD_M"]
    - pfizer_harmonized_final["FY2021_USD_M"]
)

# Identify products with the largest positive revenue change
pfizer_growth_drivers = (
    pfizer_harmonized_final
    .dropna(subset=["FY2021_USD_M", "FY2025_USD_M"])
    .sort_values("Revenue_Change_USD_M", ascending=False)
    .head(10)
)

display(
    pfizer_growth_drivers[
        [
            "Category",
            "Product_or_Category",
            "FY2021_USD_M",
            "FY2025_USD_M",
            "Revenue_Change_USD_M"
        ]
    ]
)

,Category,Product_or_Category,FY2021_USD_M,FY2025_USD_M,Revenue_Change_USD_M
59,Specialty Care,Vyndaqel family,2015.0,6380.0,4365.0
35,Primary Care,Paxlovid,76.0,2362.0,2286.0
37,Primary Care,Prevnar family,5272.0,6494.0,1222.0
1,Oncology,All other Oncology,238.0,1127.0,889.0
10,Oncology,Lorbrena,266.0,1023.0,757.0
6,Oncology,Braftovi/Mektovi,342.0,716.0,374.0
26,Primary Care,All other Primary Care,2604.0,2860.0,256.0
2,Oncology,Aromasin,211.0,450.0,239.0
45,Specialty Care,Cresemba,142.0,349.0,207.0
32,Primary Care,FSME-IMMUN/TicoVac,185.0,319.0,134.0


In [70]:
# Identify products with the largest absolute revenue decline
pfizer_declining_products = (
    pfizer_harmonized_final
    .dropna(subset=["FY2021_USD_M", "FY2025_USD_M"])
    .sort_values("Revenue_Change_USD_M", ascending=True)
    .head(10)
)

display(
    pfizer_declining_products[
        [
            "Category",
            "Product_or_Category",
            "FY2021_USD_M",
            "FY2025_USD_M",
            "Revenue_Change_USD_M"
        ]
    ]
)

,Category,Product_or_Category,FY2021_USD_M,FY2025_USD_M,Revenue_Change_USD_M
60,Specialty Care,Xeljanz,2455.0,1087.0,-1368.0
8,Oncology,Ibrance,5437.0,4122.0,-1315.0
41,Specialty Care,All other Specialty Care,2830.0,1588.0,-1242.0
46,Specialty Care,Enbrel (Outside the U.S. and Canada),1185.0,627.0,-558.0
9,Oncology,Inlyta,1002.0,923.0,-79.0
50,Specialty Care,Inflectra,657.0,646.0,-11.0
48,Specialty Care,Genotropin,389.0,446.0,57.0
5,Oncology,Bosulif,540.0,611.0,71.0
63,Specialty Care,Zithromax,278.0,399.0,121.0
32,Primary Care,FSME-IMMUN/TicoVac,185.0,319.0,134.0


In [71]:
# Calculate the top 5 products' share of FY2025 portfolio revenue

top_5_revenue = (
    pfizer_harmonized_final
    .nlargest(5, "FY2025_USD_M")["FY2025_USD_M"]
    .sum()
)

total_portfolio_revenue = reported_totals[
    "FY2025_USD_M"
].sum()

top_5_share = (
    top_5_revenue / total_portfolio_revenue
) * 100

print("Top 5 revenue:", round(top_5_revenue, 2), "USD million")
print("Total portfolio revenue:", total_portfolio_revenue, "USD million")
print("Top 5 concentration:", round(top_5_share, 2), "%")

Top 5 revenue: 29324.0 USD million
Total portfolio revenue: 61200 USD million
Top 5 concentration: 47.92 %
